# nb05 — RNA concordance across independent captures

## Scientific question
For one donor, do CITE-seq and Multiome recover the same cell-type-specific genes
and biological pathways, sufficiently consistently to motivate multimodal follow-up?

## Analysis strategy
Compare independent cell-type mean profiles on one shared gene universe, inspect
specificity-rank and top-gene agreement, and compare pathway NES with statistical
support. Repeat expression comparisons with equal cell counts and inspect shared
sites. Shortlist replicated positive programs by effect, shared leading edge and
cell-type diversity. Explore a shared uncorrected PCA of population means only.

Run nb04 first. This notebook reads its saved outputs and verifies hashes to reject
mixed/stale runs. It never aligns CITE and Multiome cells by row or barcode.

In [ ]:
from pathlib import Path
import sys
import subprocess

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # Same Drive root as nb01. No changes to the original benchmark files.
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'anndata>=0.11,<0.13'])
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = Path('/content/drive/MyDrive/multiomics-relationship-modeling')
else:
    BASE_PATH = Path.cwd().resolve()
    if not (BASE_PATH / 'src').is_dir():
        BASE_PATH = BASE_PATH.parent

if not (BASE_PATH / 'src/single_donor_workflow.py').is_file():
    raise FileNotFoundError('Copy the updated src/, configs/ and new notebooks to the project on Drive first.')
sys.path.insert(0, str(BASE_PATH))
import pandas as pd
from IPython.display import display, Image
from configs.paths import CITE_H5AD, MULTIOME_H5AD
from src.single_donor_io import output_dirs

OUTPUT_ROOT = BASE_PATH / 'results/single_donor'
DIRS = output_dirs(OUTPUT_ROOT)
print('Colab:', IN_COLAB, '| Project:', BASE_PATH)

In [ ]:
from src.single_donor_workflow import load_characterization, run_concordance
manifest, programs, enrichment, universe, rna_data = load_characterization(OUTPUT_ROOT)
display(manifest)
print('Donor:', manifest['donor'], '| Shared genes:', len(universe))

## Expression, ranked-gene and pathway concordance
Pearson/Spearman correlations use independent within-cell-type average RNA profiles.
Specificity-rank Spearman is computed over the complete shared eligible gene list.
Top-50/100/250 overlap uses only positive-specificity genes, with actual list lengths
reported when fewer genes qualify. No gene-correlation p-values are interpreted as
evidence across individuals.

Pathway classes use q ≤ 0.05 in both captures for concordant active/negative or
opposite-direction discordant results. A pathway significant in only one capture is
labelled capture-specific **descriptively**; a difference in significance does not
establish a significant difference. Non-significant results are unsupported, never
automatically inactive. Negative enrichment denotes relative underrepresentation.

Five balanced sampling repeats use independent draws in each capture and the same
cell count per cell type (minimum of both available counts and 1000). This is a
sensitivity analysis, not five biological replicates. One balanced draw also repeats
enrichment with the original shared gene universe and collection. Site-specific
comparisons require at least two adequately represented shared cell types.

In [ ]:
BALANCED_REPEATS = 5
BALANCED_CAP = 1000
expression_comparison, pathway_comparison, followup, findings = run_concordance(
    OUTPUT_ROOT, repeats=BALANCED_REPEATS, cap=BALANCED_CAP)
display(expression_comparison)
display(pd.read_csv(DIRS['rna_concordance'] / 'top_gene_overlap.csv'))
display(pathway_comparison[['cell_type', 'pathway', 'NES_cite', 'NES_multiome', 'q_value_cite', 'q_value_multiome', 'concordance']])
display(pd.read_csv(DIRS['rna_concordance'] / 'pathway_correlations.csv'))
for name in ['expression_scatter', 'pathway_nes_scatter', 'pathway_concordance_heatmap']:
    figure = DIRS['rna_concordance'] / 'figures' / f'{name}.png'
    if figure.exists() and (name == 'expression_scatter' or not pathway_comparison.empty):
        display(Image(filename=str(figure)))

## Unequal cell numbers and donor/site composition
Compare the balanced ranges below to the complete-profile estimates above. A range
across draws describes sampling sensitivity, not confidence across donors. Shared
site analyses change the comparator pool when some populations are absent; inspect
their coverage table. If no shared site is testable, site confounding remains
unresolved. With one selected donor, donor effects cannot be estimated.

In [ ]:
sensitivity = pd.read_csv(DIRS['rna_concordance'] / 'balanced_expression_sensitivity.csv')
display(sensitivity.groupby('cell_type')[['pearson', 'spearman', 'specificity_rank_spearman']].agg(['min', 'median', 'max']))
display(pd.read_csv(DIRS['rna_concordance'] / 'balanced_pathway_sensitivity.csv'))
display(pd.read_csv(DIRS['rna_concordance'] / 'site_sensitivity_coverage.csv'))
display(pd.read_csv(DIRS['rna_concordance'] / 'site_expression_sensitivity.csv'))

## Candidate RNA programs for follow-up
Require replicated positive enrichment, at least two shared leading-edge genes and
positive NES in the balanced sensitivity run. A round-robin across cell types ranks
by the weaker NES, limits redundant leading edges and yields **up to ten** candidates;
it does not force five when evidence is insufficient. No candidate is chosen solely
by p-value. Inspect biological relevance and pathway redundancy explicitly: a list
of statistically supported pathways is not automatically a diverse biological panel.

The saved `biological_review` field remains pending. Use the table and leading edges
to choose interpretable programs before implementing nb06. Scarce Hallmark coverage
of antigen presentation, cytotoxicity or stemness may justify a later, separately
documented Reactome/GO run; do not invent these findings or combine mismatched runs.

In [ ]:
display(followup[['cell_type', 'pathway', 'NES_cite', 'NES_multiome', 'shared_leading_edge',
                  'balanced_direction_supported', 'selected_for_followup', 'reason', 'biological_review']])
print('Candidate programs:', int(followup.selected_for_followup.sum()))

## Exploratory RNA bridge assessment
This PCA uses a common gene basis and uncorrected population mean RNA, with equal
weight for every dataset × cell-type centroid. It does not compare unrelated PCA axes
from the original objects. Inspect whether equivalent cell types are near each other
and whether capture identity separates them. Centroid agreement does not establish
cell-level latent alignment. No aggressive batch correction, latent-space transfer,
multi-donor pseudobulk or full integration is implemented.

In [ ]:
display(pd.read_csv(DIRS['rna_concordance'] / 'shared_centroid_pca.csv'))
display(Image(filename=str(DIRS['rna_concordance'] / 'figures/shared_centroid_pca.png')))

## Main findings
The quantitative report below is computed from this run. Use correlations, rank
overlap, pathway coverage, balanced draws, site checks and leading edges jointly to
assess the RNA bridge. No universal pass threshold or biological conclusion is
pre-filled. **Review these results before implementing ATAC and protein analyses.**

In [ ]:
display(findings)
print('Saved tables, figures and provenance:', OUTPUT_ROOT)

## Limitations
- One donor cannot establish reproducibility across individuals or donor effects.
- Separate CITE/Multiome cells prohibit cross-capture cell-level association claims.
- Broad lineage agreement can conceal subtype composition and capture differences.
- Correlation can be driven by abundant/housekeeping genes: inspect specificity and
  pathway agreement alongside mean expression, and check the technical gene flags.
- Gene-set permutation p-values ignore gene correlation; q-values describe the
  chosen exploratory test family, not condition or individual-level inference.
- Balanced subsampling cannot correct unmeasured technical or site confounders.
- Negative specificity is not biological inactivity; missing evidence is not absence.
- Subsequent protein coverage is limited, and ATAC gene activity approximates
  regulatory accessibility. Neither snapshot establishes causality or temporal priming.
- nb06–nb08 are deliberately deferred until this RNA milestone runs and is reviewed
  on the real benchmark files in Drive, following the requested incremental sequence.